# 🔬 QMDD: Adversarial Robustness of Quantized Deepfake Detection
**Paper:** Unlocking the Potential of Lightweight Quantized Models for Deepfake Detection (IJCAI 2025)

**Dataset:** CIFAR-10 (auto-downloaded) — Automobile=Real / Horse=Fake

> No external dataset needed. CIFAR-10 loads automatically via torchvision.

**Runtime:** `Runtime → Change runtime type → T4 GPU`

---
## 📦 Cell 1 — Check GPU

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU — go to Runtime -> Change runtime type -> T4 GPU')


---
## 🛠️ Cell 2 — Install Dependencies

In [ ]:
!pip install -q scipy scikit-learn tensorboardX
print('Done.')


---
## 🔁 Cell 3 — Clone QMDD Repository

In [ ]:
import os
if not os.path.exists('/content/QMDD'):
    !git clone https://github.com/rstao-bjtu/QMDD.git /content/QMDD
else:
    !cd /content/QMDD && git pull
os.chdir('/content/QMDD')
print('Working dir:', os.getcwd())
!ls


---
## 📁 Cell 4 — Load CIFAR-10 as Real/Fake Binary Dataset

- **Real (label=0):** Automobile images
- **Fake (label=1):** Horse images


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

REAL_CLASS = 1   # automobile
FAKE_CLASS = 7   # horse
DATA_DIR   = '/content/cifar10_data'

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_train = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=transform)
full_test  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform)

def make_binary_subset(dataset, real_cls, fake_cls, max_per_class=500):
    targets  = np.array(dataset.targets)
    real_idx = np.where(targets == real_cls)[0][:max_per_class]
    fake_idx = np.where(targets == fake_cls)[0][:max_per_class]
    indices  = np.concatenate([real_idx, fake_idx])
    np.random.shuffle(indices)
    sub = Subset(dataset, indices)
    sub.dataset.targets = [
        0 if t == real_cls else (1 if t == fake_cls else t)
        for t in dataset.targets
    ]
    return sub

train_subset = make_binary_subset(full_train, REAL_CLASS, FAKE_CLASS, max_per_class=1000)
test_subset  = make_binary_subset(full_test,  REAL_CLASS, FAKE_CLASS, max_per_class=200)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_subset,  batch_size=32, shuffle=False, num_workers=2)

print(f'Train samples : {len(train_subset)}')
print(f'Test  samples : {len(test_subset)}')
print('Classes       : 0=Automobile (real), 1=Horse (fake)')
print('Dataset ready!')


---
## 🖼️ Cell 5 — Visualize Dataset Samples

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

imgs, labels = next(iter(test_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('CIFAR-10 Binary Dataset  |  Top: Real (Automobile)   Bottom: Fake (Horse)',
             fontsize=12, fontweight='bold')

real_imgs = imgs[labels == 0][:8]
fake_imgs = imgs[labels == 1][:8]

for col in range(8):
    for row, (row_imgs, title) in enumerate([(real_imgs, 'Real'), (fake_imgs, 'Fake')]):
        if col < len(row_imgs):
            img = row_imgs[col].permute(1,2,0).numpy()
            img = np.clip(img * std + mean, 0, 1)
            axes[row][col].imshow(img)
            axes[row][col].set_title(title, fontsize=8)
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('/content/dataset_samples.png', dpi=100, bbox_inches='tight')
plt.show()


---
## 🏗️ Cell 6 — Define & Build the Quantized Model

The `QMDDModel` class and `SLRQuantizer` are defined here **globally** so they are
available throughout the notebook (required for saving/loading weights correctly).

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ── SLR Quantizer (from QMDD paper) ───────────────────────────────
class SLRQuantizer(nn.Module):
    """Shifted Logarithmic Redistribution Quantizer.
    Unfolds near-zero activations via log mapping before uniform quantization.
    tau: learnable layer-wise offset — higher tau = smoother redistribution.
    """
    def __init__(self, bits=2, tau_init=1.0):
        super().__init__()
        self.bits     = bits
        self.n_levels = 2 ** bits
        self.tau      = nn.Parameter(torch.tensor(float(tau_init)))
        self.epsilon  = 1.0

    def forward(self, x):
        # Compute offset: off = min(max(|X|), epsilon) + tau
        off   = torch.clamp(x.abs().max(), min=self.epsilon) + self.tau
        x_log = torch.log(x + off)
        x_min = x_log.min()
        x_max = x_log.max()
        if x_max == x_min:
            return x
        scale = (x_max - x_min) / (self.n_levels - 1)
        x_q   = torch.round((x_log - x_min) / scale) * scale + x_min
        return torch.exp(x_q) - off


# ── QMDD Model ─────────────────────────────────────────────────────
class QMDDModel(nn.Module):
    """Lightweight binary deepfake detector with SLR quantization.
    Uses ResNet-18 layers 1-3 as backbone (mimics QMDD ResNet-23 CQB structure).
    """
    def __init__(self, bits=2, tau_init=1.0):
        super().__init__()
        backbone     = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.stem    = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1  = backbone.layer1
        self.layer2  = backbone.layer2
        self.layer3  = backbone.layer3
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.slr1    = SLRQuantizer(bits=bits, tau_init=tau_init)
        self.slr2    = SLRQuantizer(bits=bits, tau_init=tau_init)
        self.slr3    = SLRQuantizer(bits=bits, tau_init=tau_init)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.slr1(self.layer1(x))
        x = self.slr2(self.layer2(x))
        x = self.slr3(self.layer3(x))
        x = self.pool(x)
        return self.classifier(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── Instantiate ────────────────────────────────────────────────────
model = QMDDModel(bits=2, tau_init=1.0).cuda()
print(f'Model parameters : {model.count_params():,}')
print(f'Quantization     : 2-bit SLR Quantizer (tau=1.0)')
print('Model ready!')


---
## 🏋️ Cell 7 — Train the Quantized Model

> **Fix applied:** We save `model.state_dict()` instead of the full model object.
> This is compatible with PyTorch 2.6+ and avoids the `UnpicklingError`.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score

EPOCHS    = 10
LR        = 1e-4
# Save path for state_dict only (safe & PyTorch 2.6+ compatible)
SAVE_PATH = '/content/qmdd_cifar10_state.pth'

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
criterion = nn.BCEWithLogitsLoss()

train_losses, train_accs, val_accs = [], [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    epoch_loss, y_true_all, y_pred_all = 0.0, [], []
    for imgs, labels in train_loader:
        imgs   = imgs.cuda()
        labels = labels.float().cuda()
        optimizer.zero_grad()
        out  = model(imgs).squeeze(1)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        y_true_all.extend(labels.cpu().tolist())
        y_pred_all.extend(out.sigmoid().detach().cpu().tolist())

    train_acc = accuracy_score(np.array(y_true_all), np.array(y_pred_all) > 0.5)
    avg_loss  = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    train_accs.append(train_acc)

    # ── Validate ──
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            out = model(imgs.cuda()).sigmoid().squeeze().cpu().tolist()
            yt.extend(labels.tolist())
            yp.extend(out if isinstance(out, list) else [out])
    val_acc = accuracy_score(np.array(yt), np.array(yp) > 0.5)
    val_accs.append(val_acc)
    scheduler.step()
    print(f'Epoch [{epoch:02d}/{EPOCHS}]  Loss: {avg_loss:.4f}  '
          f'Train Acc: {train_acc*100:.1f}%  Val Acc: {val_acc*100:.1f}%')

# ── Save state_dict ONLY (PyTorch 2.6+ safe) ──────────────────────
torch.save(model.state_dict(), SAVE_PATH)
print(f'\nState dict saved to {SAVE_PATH}')
print('(state_dict save avoids UnpicklingError on PyTorch 2.6+)')


---
## 📈 Cell 8 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('QMDD Training on CIFAR-10 (2-bit SLR Quantized ResNet-18)', fontweight='bold')

axes[0].plot(range(1, EPOCHS+1), train_losses, 'o-', color='#E53935', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training Loss'); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS+1), [a*100 for a in train_accs], 'o-', color='#1E88E5', label='Train', lw=2)
axes[1].plot(range(1, EPOCHS+1), [a*100 for a in val_accs],   's--', color='#43A047', label='Val',   lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Train vs Validation Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 📊 Cell 9 — Full Evaluation (Acc, AP, Confusion Matrix)

In [ ]:
import torch, numpy as np
from sklearn.metrics import (accuracy_score, average_precision_score,
                              confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
y_true, y_pred_prob = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        probs = model(imgs.cuda()).sigmoid().squeeze().cpu().tolist()
        y_true.extend(labels.tolist())
        y_pred_prob.extend(probs if isinstance(probs, list) else [probs])

y_true      = np.array(y_true)
y_pred_prob = np.array(y_pred_prob)
y_pred      = (y_pred_prob > 0.5).astype(int)

CLEAN_ACC = accuracy_score(y_true, y_pred)
CLEAN_AP  = average_precision_score(y_true, y_pred_prob)
cm        = confusion_matrix(y_true, y_pred)

print(f'Clean Accuracy : {CLEAN_ACC*100:.2f}%')
print(f'Average Prec.  : {CLEAN_AP*100:.2f}%')
print()
print(classification_report(y_true, y_pred, target_names=['Real (auto)', 'Fake (horse)']))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Clean Test Set)')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 🛡️ Cell 10 — Define Adversarial Attacks (FGSM & PGD)

In [ ]:
import torch
import torch.nn as nn

def fgsm_attack(model, images, labels, epsilon=8/255):
    images  = images.clone().detach().cuda().requires_grad_(True)
    labels  = labels.clone().detach().float().cuda()
    out     = model(images).squeeze(1)
    loss    = nn.BCEWithLogitsLoss()(out, labels)
    model.zero_grad(); loss.backward()
    adv = images + epsilon * images.grad.sign()
    return torch.clamp(adv, 0, 1).detach()

def pgd_attack(model, images, labels, epsilon=8/255, alpha=2/255, iters=20):
    orig   = images.clone().detach().cuda()
    labels = labels.clone().detach().float().cuda()
    adv    = orig + torch.empty_like(orig).uniform_(-epsilon, epsilon)
    adv    = torch.clamp(adv, 0, 1).detach()
    for _ in range(iters):
        adv.requires_grad_(True)
        out  = model(adv).squeeze(1)
        loss = nn.BCEWithLogitsLoss()(out, labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            adv   = adv + alpha * adv.grad.sign()
            delta = torch.clamp(adv - orig, -epsilon, epsilon)
            adv   = torch.clamp(orig + delta, 0, 1)
    return adv.detach()

print('FGSM and PGD attack functions ready.')


---
## 🔬 Cell 11 — Adversarial Robustness Evaluation

In [ ]:
import torch, numpy as np
from sklearn.metrics import accuracy_score

model.eval()
epsilons    = [4/255, 8/255, 16/255]
rob_results = []

print(f'{"Attack":<8} {"Epsilon":<10} {"Acc (%)":>10} {"Drop (%)":>10}')
print('-' * 42)
print(f'{"Clean":<8} {"--":<10} {CLEAN_ACC*100:>9.1f}% {0.0:>9.1f}%')

for eps in epsilons:
    fgsm_yt, fgsm_yp = [], []
    pgd_yt,  pgd_yp  = [], []

    for imgs, labels in test_loader:
        adv_f = fgsm_attack(model, imgs, labels, epsilon=eps)
        with torch.no_grad():
            pf = model(adv_f).sigmoid().squeeze().cpu().tolist()
        fgsm_yt.extend(labels.tolist())
        fgsm_yp.extend(pf if isinstance(pf, list) else [pf])

        adv_p = pgd_attack(model, imgs, labels, epsilon=eps, iters=10)
        with torch.no_grad():
            pp = model(adv_p).sigmoid().squeeze().cpu().tolist()
        pgd_yt.extend(labels.tolist())
        pgd_yp.extend(pp if isinstance(pp, list) else [pp])

    fa = accuracy_score(np.array(fgsm_yt), np.array(fgsm_yp) > 0.5)
    pa = accuracy_score(np.array(pgd_yt),  np.array(pgd_yp)  > 0.5)
    rob_results.append({'eps': eps, 'fgsm_acc': fa, 'pgd_acc': pa})
    print(f'{"FGSM":<8} {eps*255:.0f}/255 {"":>5} {fa*100:>9.1f}% {(CLEAN_ACC-fa)*100:>9.1f}%')
    print(f'{"PGD":<8} {eps*255:.0f}/255 {"":>5} {pa*100:>9.1f}% {(CLEAN_ACC-pa)*100:>9.1f}%')
    print()

print('Adversarial robustness evaluation done.')


---
## 💡 Cell 12 — Adversarial-SLR: τ (tau) Sweep

**Research contribution:** We test whether tuning the SLR quantizer offset `tau`
improves robustness against adversarial attacks.

- Higher `tau` → wider log redistribution → smoother decision boundaries → potentially more robust
- We sweep `tau` from 0 to 8 and measure the clean vs. adversarial accuracy trade-off.

> **Fix:** Uses `model.state_dict()` save/load with `weights_only=True` — fully compatible with PyTorch 2.6+.
> A fresh `QMDDModel` instance is created per tau value, weights are loaded via `load_state_dict()`,
> then tau is overridden before evaluation.

In [ ]:
import copy, torch, numpy as np
from sklearn.metrics import accuracy_score


def set_tau(m, val):
    """Override all SLR quantizer tau parameters in the model."""
    n = 0
    for mod in m.modules():
        if hasattr(mod, 'tau') and isinstance(mod.tau, torch.nn.Parameter):
            with torch.no_grad():
                mod.tau.fill_(float(val))
            n += 1
    return n


def load_model_with_tau(state_dict_path, tau_value):
    """Safe model loading: instantiate class first, then load weights.
    This avoids the PyTorch 2.6+ UnpicklingError that occurs with torch.load(full_model).
    """
    m = QMDDModel(bits=2, tau_init=1.0).cuda()           # fresh instance
    state = torch.load(state_dict_path, weights_only=True)  # safe load
    m.load_state_dict(state)                              # restore weights
    n = set_tau(m, tau_value)                             # override tau
    print(f'  tau={tau_value:.1f} -> {n} SLR layers updated', end='  ')
    return m


def quick_eval(m, loader, epsilon=8/255, max_batches=8):
    """Evaluate clean + PGD accuracy over a subset of the loader."""
    m.eval()
    yt_c, yp_c = [], []
    yt_p, yp_p = [], []
    for i, (imgs, labels) in enumerate(loader):
        if i >= max_batches:
            break
        with torch.no_grad():
            pc = m(imgs.cuda()).sigmoid().squeeze().cpu().tolist()
        yt_c.extend(labels.tolist())
        yp_c.extend(pc if isinstance(pc, list) else [pc])

        adv = pgd_attack(m, imgs, labels.float(), epsilon=epsilon, iters=10)
        with torch.no_grad():
            pp = m(adv).sigmoid().squeeze().cpu().tolist()
        yt_p.extend(labels.tolist())
        yp_p.extend(pp if isinstance(pp, list) else [pp])

    ca = accuracy_score(np.array(yt_c), np.array(yp_c) > 0.5)
    pa = accuracy_score(np.array(yt_p), np.array(yp_p) > 0.5)
    return ca, pa


# ── Tau sweep ──────────────────────────────────────────────────────
tau_values  = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]
tau_results = []

print(f'{"tau":>6} | {"Clean Acc":>10} | {"PGD Acc":>10} | {"Robustness Gap":>14}')
print('-' * 52)

for tau in tau_values:
    m = load_model_with_tau(SAVE_PATH, tau)     # safe PyTorch 2.6+ load
    ca, pa = quick_eval(m, test_loader)
    gap = ca - pa
    tau_results.append({'tau': tau, 'clean_acc': ca, 'pgd_acc': pa, 'gap': gap})
    print(f'{tau:>6.1f} | {ca*100:>9.1f}% | {pa*100:>9.1f}% | {gap*100:>13.1f}%')

print()
best = min(tau_results, key=lambda x: x['gap'])
print(f'Best tau for robustness: {best["tau"]}  (robustness gap = {best["gap"]*100:.1f}%)')


---
## 📊 Cell 13 — Full Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

taus      = [r['tau']       for r in tau_results]
clean_acc = [r['clean_acc'] * 100 for r in tau_results]
pgd_acc   = [r['pgd_acc']   * 100 for r in tau_results]
gaps      = [r['gap']       * 100 for r in tau_results]
eps_lbl   = [f'{r["eps"]*255:.0f}/255' for r in rob_results]
fgsm_accs = [r['fgsm_acc'] * 100 for r in rob_results]
pgd_accs_ = [r['pgd_acc']  * 100 for r in rob_results]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('QMDD Adversarial Robustness Analysis on CIFAR-10\n'
             '2-bit SLR Quantized ResNet-18  |  Real=Automobile / Fake=Horse',
             fontsize=13, fontweight='bold')

# (A) Tau sweep
axes[0][0].plot(taus, clean_acc, 'o-', color='#1E88E5', label='Clean', lw=2, ms=8)
axes[0][0].plot(taus, pgd_acc,   's--', color='#E53935', label='PGD (e=8/255)', lw=2, ms=8)
axes[0][0].set_xlabel('SLR tau'); axes[0][0].set_ylabel('Accuracy (%)')
axes[0][0].set_title('(A) Adversarial-SLR: Tau vs Accuracy')
axes[0][0].legend(); axes[0][0].grid(alpha=0.3)
axes[0][0].yaxis.set_major_formatter(mtick.PercentFormatter())

# (B) Robustness gap
colors = ['#43A047' if g <= np.mean(gaps) else '#FB8C00' for g in gaps]
axes[0][1].bar(taus, gaps, color=colors, width=0.6, alpha=0.85)
axes[0][1].axhline(np.mean(gaps), color='red', ls='--', lw=1.5,
                   label=f'Mean gap: {np.mean(gaps):.1f}%')
axes[0][1].set_xlabel('SLR tau'); axes[0][1].set_ylabel('Accuracy Drop (%)')
axes[0][1].set_title('(B) Robustness Gap (Clean - PGD)\nGreen = below average (more robust)')
axes[0][1].legend(); axes[0][1].grid(axis='y', alpha=0.3)
axes[0][1].yaxis.set_major_formatter(mtick.PercentFormatter())

# (C) FGSM vs PGD across epsilons
x = np.arange(len(eps_lbl))
axes[1][0].bar(x - 0.175, fgsm_accs, 0.35, label='FGSM', color='#7B1FA2', alpha=0.85)
axes[1][0].bar(x + 0.175, pgd_accs_, 0.35, label='PGD',  color='#E53935', alpha=0.85)
axes[1][0].axhline(CLEAN_ACC*100, color='#1E88E5', ls='--', lw=2,
                   label=f'Clean: {CLEAN_ACC*100:.1f}%')
axes[1][0].set_xticks(x); axes[1][0].set_xticklabels(eps_lbl)
axes[1][0].set_xlabel('Epsilon'); axes[1][0].set_ylabel('Accuracy (%)')
axes[1][0].set_title('(C) Attack Accuracy vs Epsilon')
axes[1][0].legend(); axes[1][0].grid(axis='y', alpha=0.3)
axes[1][0].yaxis.set_major_formatter(mtick.PercentFormatter())

# (D) Training history
axes[1][1].plot(range(1, EPOCHS+1), [a*100 for a in train_accs], 'o-',
                color='#1E88E5', label='Train', lw=2)
axes[1][1].plot(range(1, EPOCHS+1), [a*100 for a in val_accs],   's--',
                color='#43A047', label='Val', lw=2)
axes[1][1].set_xlabel('Epoch'); axes[1][1].set_ylabel('Accuracy (%)')
axes[1][1].set_title('(D) Training History')
axes[1][1].legend(); axes[1][1].grid(alpha=0.3)
axes[1][1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('/content/full_results.png', dpi=130, bbox_inches='tight')
plt.show()
print('Plot saved to /content/full_results.png')


---
## 📋 Cell 14 — Results Summary Tables

In [ ]:
import pandas as pd

rows = [{'Attack': 'Clean', 'Epsilon': '--',
         'Acc (%)': f'{CLEAN_ACC*100:.1f}',
         'AP (%)':  f'{CLEAN_AP*100:.1f}',
         'Drop (%)': '0.0'}]
for r in rob_results:
    rows.append({'Attack': 'FGSM', 'Epsilon': f'{r["eps"]*255:.0f}/255',
                 'Acc (%)': f'{r["fgsm_acc"]*100:.1f}', 'AP (%)': '--',
                 'Drop (%)': f'{(CLEAN_ACC - r["fgsm_acc"])*100:.1f}'})
    rows.append({'Attack': 'PGD',  'Epsilon': f'{r["eps"]*255:.0f}/255',
                 'Acc (%)': f'{r["pgd_acc"]*100:.1f}',  'AP (%)': '--',
                 'Drop (%)': f'{(CLEAN_ACC - r["pgd_acc"])*100:.1f}'})

df_rob = pd.DataFrame(rows)
df_tau = pd.DataFrame(tau_results)
df_tau['clean_acc'] = df_tau['clean_acc'].map(lambda x: f'{x*100:.1f}%')
df_tau['pgd_acc']   = df_tau['pgd_acc'].map(lambda x:   f'{x*100:.1f}%')
df_tau['gap']       = df_tau['gap'].map(lambda x:       f'{x*100:.1f}%')
df_tau.columns      = ['tau', 'Clean Acc', 'PGD Acc', 'Robustness Gap']

print('=' * 55)
print('   ADVERSARIAL ROBUSTNESS')
print('=' * 55)
print(df_rob.to_string(index=False))
print()
print('=' * 55)
print('   ADVERSARIAL-SLR tau SWEEP  (PGD e=8/255)')
print('=' * 55)
print(df_tau.to_string(index=False))


---
## 💾 Cell 15 — Save & Download All Results

In [ ]:
import os, shutil
from google.colab import files

SAVE_DIR = '/content/qmdd_results'
os.makedirs(SAVE_DIR, exist_ok=True)

df_rob.to_csv(f'{SAVE_DIR}/adversarial_robustness.csv', index=False)
df_tau.to_csv(f'{SAVE_DIR}/tau_sweep.csv', index=False)

for fname in ['full_results.png', 'training_curves.png',
              'confusion_matrix.png', 'dataset_samples.png']:
    src = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, SAVE_DIR)

shutil.copy(SAVE_PATH, SAVE_DIR)
shutil.make_archive('/content/qmdd_results', 'zip', SAVE_DIR)
files.download('/content/qmdd_results.zip')
print('Download started — check your browser for qmdd_results.zip')


---
## 📝 Summary & Notes

| Component | Detail |
|---|---|
| **Fix** | `state_dict` save/load + `weights_only=True` — no more `UnpicklingError` |
| **Model** | ResNet-18 backbone (layers 1-3) with 2-bit SLR Quantization |
| **Dataset** | CIFAR-10: Automobile (real=0) vs Horse (fake=1) — auto-downloaded |
| **Training** | 10 epochs, Adam lr=1e-4, BCEWithLogitsLoss |
| **Attacks** | FGSM (single-step), PGD (20-step) at eps=4,8,16/255 |
| **Adversarial-SLR** | tau swept 0.0 to 8.0 — smaller gap = more robust |

**Reference:** Tao et al., *Unlocking the Potential of Lightweight Quantized Models for Deepfake Detection*, IJCAI 2025.
